In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


In [ ]:
#Display/Plotting defaults
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 100
SEASON_ORDER = ['Kharif', 'Rabi', 'Zaid']  # chronological cropping seasons in India

In [ ]:
DATA_PATH = 'seasonal_agriculture_performance_dataset.csv'  # path to the data folder
df = pd.read_csv(DATA_PATH)  # read the dataset into a pandas DataFrame
df['Season'] = pd.Categorical(df['Season'],categories=SEASON_ORDER,ordered=True) #set the order of the cropping seasons for plotting purposes
print(f"Data loaded successfully.Dataset contains {df.shape[0]} rows and {df.shape[1]} columns.")
df.head() #Display the first few rows of the dataset
df.info() #Display th summary of the dataset including data types and non-null counts
df.describe().T #Deisplay the summanry characteristics of the dataset's numerical columns


In [ ]:
categorical_cols = ['State', 'District', 'Crop', 'Season', 'Irrigation_Method']
for c in categorical_cols:
       print(f"{c} ({df[c].nunique()} unique): {sorted(df[c].dropna().unique().tolist())}\n")

In [ ]:
missing = df.isnull().sum()
missing =missing[missing>0].sort_values(ascending=False)

missing_pct = (missing / len(df)* 100),round(2)
pd.DataFrame({'Missing Values': missing, 'Percentage': missing_pct})


In [ ]:
for col in ['Rainfall_mm' , 'Soil_Moisture_pct', 'Yield_Tonnes_Ha']:
    df[col] = df.groupby('Season' , observed = True)[col].transform(lambda x: x.fillna(x.median(x)))

print("Remaining missing values after imputation:", df.isnull().sum().sum())

dupes  = df.duplicated().sum()
print(f"Duplicate rows in the dataset:"{dupes})
if dupes:
    df = df.drop_duplicates()
    print(f"Dropped Duplicates")

In [ ]:
def iqr_outlier_summary(frame,cols):
    rows = []
    for c in cols:
        q1,q3 = frame[c].quantile([0.25,0.75])
        iqr = q3-q1
        lo,hi = q1-1.5*iqr, q3+1.5*iqr
        n_out = ((frame[c]<lo)) | ((frame[c]>hi)).sum()
        rows.append([c, round(lo, 2), round(hi, 2), n_out, round(n_out / len(frame) * 100, 1)])
    return pd.DataFrame(rows, columns=['column', 'lower_bound', 'upper_bound', 'n_outliers', 'pct_outliers'])
check_cols = ['Yield_Tonnes_Ha', 'Rainfall_mm', 'Profit_INR' , 'Water_used_m3']
iqr_outlier_summary(df, check_cols)


In [ ]:
for col in ['Yield_Tonnes_Ha', 'Production_Tonnes']:
    lo, hi = df[col].quantile([0.01, 0.99])
    df[col] = df[col].clip(lo, hi)
 
df[['Yield_Tonnes_Ha', 'Production_Tonnes']].describe().T[['min', 'max', 'mean']]


In [ ]:
df['Product_Margin_pct'] = (df['Profit_INR'] / df['Production_Tonnes']) * 100
df['Cost per Tonne'] = df['Cost_INR'] / df['Production_Tonnes']
df['Is_Profitable'] = df['Profit_INR'] > 0

df[['Product_Margin_pct', 'Cost per Tonne', 'Is_Profitable']].describe(include='all').T

In [ ]:
fig , axes = plt.subplots(1,2,figsize=(12,5))

season_counts = df['Season'].value_counts().reindex(SEASON_ORDER)
sns.barplot(x=season_counts.index,y = season_counts.values,ax = axes[0])
axes[0].set_title('Number of farm records per season')
axes[0].set_ylabel('Count')

crop_season = pd.crosstab(df['Crop'],df['Season'])
sns.heatmap(crop_season,annot=True,fmt='d',cmap='YlGnBu',ax=axes[1])
axes[1].set_title('Crop x Season Record Counts')
 
plt.tight_layout()
plt.show()
 


In [ ]:
env_cols = ['Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct', 'Sunlight_Hours_Day', 'Soil_pH', 'Soil_Moisture_pct']
 
fig, axes = plt.subplots(2,3,figsize=(16,9))
for ax, col in zip(axes.flatten(),env_cols):
    sns.boxplot(data=df,x='Season', y=col, order=SEASON_ORDER, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

df.groupby('Season', observed=True)[env_cols].mean().round(2)
 


In [ ]:
resource_cols = ['Fertilizer_kg_ha', 'Pesticide_Litre_ha', 'Water_Used_m3', 'Nitrogen_kg_ha', 'Phosphorus_kg_ha', 'Potassium_kg_ha']
 
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, resource_cols):
    sns.boxplot(data=df, x='Season', y=col, order=SEASON_ORDER, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()
 
df.groupby('Season', observed=True)[resource_cols].mean().round(2)


In [ ]:
irrigation_season = pd.crosstab(df['Season'], df['Irrigation_Method'], margins=True, margins_name='Total')
irrigation_season.round(2)

In [ ]:
econ_cols = ['Yield_Tonnes_Ha', 'Production_Tonnes', 'Revenue_INR', 'Total_Cost_INR', 'Profit_INR', 'Profit_Margin_pct']
 
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, econ_cols):
    sns.boxplot(data=df, x='Season', y=col, order=SEASON_ORDER, ax=ax, showfliers=False)
    ax.set_title(col + ' (outliers hidden for readability)')
plt.tight_layout()
plt.show()
 
season_econ = df.groupby('Season', observed=True)[econ_cols].mean().round(2)
season_econ
 


In [ ]:
profitable_share = df.groupby('Season', observed=True)['Is_Profitable'].mean().round(2) * 100
fig, ax = plt.subplots(figsize = (6,4))
sns.barplot(x=profitable_share.index,y=profitable_share.values,ax=ax)
ax.set_ylabel('% of farms with positive profit')
ax.set_title('Share of Profitable Farms by Season')
plt.tight_layout()
plt.show()
 
profitable_share.round(1)


In [ ]:
crop_season_yield = df.pivot_table(values='Yield_Tonnes_Ha', index='Crop', columns='Season', aggfunc='mean', observed=True)
crop_season_profit = df.pivot_table(values='Profit_INR', index='Crop', columns='Season', aggfunc='mean', observed=True)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(crop_season_yield.round(1), annot=True, fmt='.1f', cmap='Greens', ax=axes[0])
axes[0].set_title('Mean Yield (t/ha) — Crop x Season')
sns.heatmap(crop_season_profit.round(0), annot=True, fmt='.0f', cmap='RdYlGn', center=0, ax=axes[1])
axes[1].set_title('Mean Profit (INR) — Crop x Season')
plt.tight_layout()
plt.show()


In [ ]:
state_season_yield = df.pivot_table(values='Yield_Tonnes_Ha', index='State', columns='Season', aggfunc='mean', observed=True)
fig ,ax = plt.subplots(figsize=(7,6))
sns.heatmap(state_season_yield.round(1) , annot=True, fmt = '.lf' , cmap = 'YlGnBu', ax=ax)
ax.set_title('Mean Yield (t/ha) - State x Season')
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14,11))
sns.heatmap(corr, annot = False, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix — Numeric Features')
plt.tight_layout()
plt.show()




In [ ]:
yield_corr = corr['Yield_Tonnes_Ha'].drop('Yield_Tonnes_Ha').sort_values(key=abs, ascending=False)
profit_corr = corr['Profit_INR'].drop('Profit_INR').sort_values(key=abs, ascending=False)
 
print("Strongest correlates of Yield_Tonnes_Ha:")
print(yield_corr.head(8).round(3))
print("\nStrongest correlates of Profit_INR:")
print(profit_corr.head(8).round(3))


In [ ]:
def group_by_season(col):
    return [df.loc[df['Season'] == s, col].dropna() for s in SEASON_ORDER]
 
for col in ['Yield_Tonnes_Ha', 'Profit_INR']:
    groups = group_by_season(col)
    f_stat, p_anova = stats.f_oneway(*groups)
    h_stat, p_kw = stats.kruskal(*groups)
    print(f"--- {col} ---")
    print(f"One-way ANOVA:      F = {f_stat:.3f}, p = {p_anova:.4g}")
    print(f"Kruskal-Wallis:      H = {h_stat:.3f}, p = {p_kw:.4g}")
    sig = "statistically significant" if p_kw < 0.05 else "not statistically significant"
    print(f"=> Seasonal difference in {col} is {sig} at alpha = 0.05\n")
 
 
# %%
contingency = pd.crosstab(df['Season'], df['Irrigation_Method'])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi-square = {chi2:.2f}, dof = {dof}, p = {p_chi:.4g}")
sig = "statistically significant" if p_chi < 0.05 else "not statistically significant"
print(f"=> Association between Season and Irrigation_Method is {sig} at alpha = 0.05")
 


In [ ]:
r_overall, p_overall = stats.pearsonr(df['Rainfall_mm'], df['Yield_Tonnes_Ha'])
print(f"Overall: r = {r_overall:.3f}, p = {p_overall:.4g}")
 
for s in SEASON_ORDER:
    sub = df[df['Season'] == s]
    r, p = stats.pearsonr(sub['Rainfall_mm'], sub['Yield_Tonnes_Ha'])
    print(f"{s}: r = {r:.3f}, p = {p:.4g}")
 
 


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=df, x='Season', y='Disease_Pest_Risk_pct', order=SEASON_ORDER, ax=axes[0])
axes[0].set_title('Disease/Pest Risk % by Season')
sns.boxplot(data=df, x='Season', y='Water_Efficiency_t_per_1000m3', order=SEASON_ORDER, ax=axes[1], showfliers=False)
axes[1].set_title('Water Efficiency (t per 1000 m3) by Season')
plt.tight_layout()
plt.show()
 
df.groupby('Season', observed=True)[['Disease_Pest_Risk_pct', 'Water_Efficiency_t_per_1000m3']].mean().round(2)
 


In [ ]:
# %%
best_yield_season = season_econ['Yield_Tonnes_Ha'].idxmax()
best_profit_season = season_econ['Profit_INR'].idxmax()
worst_profit_season = season_econ['Profit_INR'].idxmin()
highest_cost_season = season_econ['Total_Cost_INR'].idxmax()
top_yield_corr = yield_corr.index[0]
top_profit_corr = profit_corr.index[0]
 
print("KEY INSIGHTS")
print("=" * 60)
print(f"1. {best_yield_season} has the highest average yield "
      f"({season_econ.loc[best_yield_season, 'Yield_Tonnes_Ha']:.2f} t/ha), while "
      f"{season_econ['Yield_Tonnes_Ha'].idxmin()} has the lowest "
      f"({season_econ['Yield_Tonnes_Ha'].min():.2f} t/ha).")
print(f"2. {best_profit_season} delivers the strongest average profit "
      f"(INR {season_econ.loc[best_profit_season, 'Profit_INR']:,.0f}/farm), while "
      f"{worst_profit_season} performs worst on average "
      f"(INR {season_econ.loc[worst_profit_season, 'Profit_INR']:,.0f}/farm).")
print(f"3. Only {profitable_share.min():.1f}%-{profitable_share.max():.1f}% of farms are profitable "
      f"depending on the season -- roughly half of all farms in this dataset post a loss, "
      f"regardless of season.")
print(f"4. {highest_cost_season} carries the highest average production cost, which does not "
      f"always correspond to the highest average profit -- cost and profitability don't move together.")
print(f"5. Among all numeric factors, '{top_yield_corr}' has the strongest linear relationship with "
      f"yield (r = {yield_corr.iloc[0]:.2f}), and '{top_profit_corr}' has the strongest relationship "
      f"with profit (r = {profit_corr.iloc[0]:.2f}).")
print(f"6. The rainfall-yield relationship (overall r = {r_overall:.2f}) is weak, suggesting yield in "
      f"this dataset is driven more by inputs (fertilizer, seed quality, irrigation method) than by "
      f"rainfall alone.")
print(f"7. The seasonal differences in yield and profit found in Section 11 are "
      f"{'statistically significant' if p_kw < 0.05 else 'not statistically significant'} "
      f"(Kruskal-Wallis p = {p_kw:.4g}), meaning season is "
      f"{'a real driver of' if p_kw < 0.05 else 'not clearly linked to'} performance differences, "
      f"not just random noise.")
print(f"8. Irrigation method usage is "
      f"{'significantly associated' if p_chi < 0.05 else 'not significantly associated'} with season "
      f"(chi-square p = {p_chi:.4g}).")
 
 
